# Fine-tune Whisper Large V3 for Taiwanese Mandarin ASR (Unsloth LoRA)

**Key fixes applied in this notebook:**
- ✅ `RuntimeError: Input type (float) and bias type (c10::Half)` → resolved by using `FastModel` which correctly patches mixed-precision
- ✅ `forced_decoder_ids` deprecated → replaced with `generation_config.language/task`
- ✅ Full fine-tuning → LoRA adapter (only ~2% of parameters trained, 50%+ VRAM savings)
- ✅ Google Drive persistence for all checkpoints and cache
- ✅ `streaming=True` → dataset never downloaded to Colab disk
- ✅ Optional custom recording interleave support

**Google Drive directory layout (auto-created):**
```
MyDrive/taiwan-whisper/
├── hf_cache/          ← HuggingFace model cache
├── checkpoints/       ← Training checkpoints
├── final_model/       ← LoRA adapters
├── merged_model/      ← Merged full model (for ct2 conversion)
└── custom_data/       ← Your own recordings (place manually)
    ├── metadata.csv
    └── rec_001.wav
```


## 1. Mount Google Drive & Configure Directories


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR       = '/content/drive/MyDrive/taiwan-whisper'
CACHE_DIR       = f'{DRIVE_DIR}/hf_cache'
CHECKPOINT_DIR  = f'{DRIVE_DIR}/checkpoints'
CUSTOM_DATA_DIR = f'{DRIVE_DIR}/custom_data'

for d in [DRIVE_DIR, CACHE_DIR, CHECKPOINT_DIR, CUSTOM_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME']            = CACHE_DIR
os.environ['HF_DATASETS_CACHE']  = f'{CACHE_DIR}/datasets'
os.environ['TRANSFORMERS_CACHE'] = f'{CACHE_DIR}/transformers'
os.environ['HF_HUB_CACHE']       = f'{CACHE_DIR}/hub'

print(f'Drive mounted. Base dir: {DRIVE_DIR}')
os.system(f'du -sh {DRIVE_DIR}')


## 2. Install Dependencies (Unsloth)


In [ ]:
%%capture
import os, re
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf 'datasets>=3.4.1' 'huggingface_hub>=0.34.0' hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install librosa soundfile evaluate jiwer torchcodec
print('Done.')


## 3. Load Model (Unsloth FastModel + LoRA)

**Why `FastModel` instead of the standard `from_pretrained`?**
- Automatically patches conv1d mixed-precision (eliminates `Input type (float) and bias type (c10::Half)` RuntimeError)
- LoRA targets only `q_proj` and `v_proj` — 50%+ VRAM savings over full fine-tuning
- `use_gradient_checkpointing='unsloth'` further reduces VRAM on A100/T4
- `task_type=None` is required for Whisper LoRA


In [ ]:
from unsloth import FastModel
from transformers import WhisperForConditionalGeneration

model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/whisper-large-v3',
    dtype=None,          # auto-detect: A100 -> bf16, T4 -> fp16
    load_in_4bit=False,  # set True to save more VRAM at slight accuracy cost
    auto_model=WhisperForConditionalGeneration,
    whisper_language='chinese',
    whisper_task='transcribe',
)

# Apply LoRA — task_type=None is required for Whisper
model = FastModel.get_peft_model(
    model,
    r=64,
    target_modules=['q_proj', 'v_proj'],
    lora_alpha=64,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    task_type=None,  # required for Whisper
)

# Use generation_config instead of deprecated forced_decoder_ids
model.generation_config.language = '<|zh|>'
model.generation_config.task = 'transcribe'
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

print('Model + LoRA ready.')


## 4. Load Taiwan ASR Dataset (Streaming Mode)


In [ ]:
from datasets import load_dataset

raw_ds = load_dataset(
    'adi-gov-tw/Taiwan-Tongues-ASR-CE-dataset-zhtw',
    'default',
    streaming=True,
)

sample = next(iter(raw_ds['train']))
print('Train columns:', list(sample.keys()))
print('Text sample:', sample.get('txt', sample.get('text', '?')))


## 5. Add Custom Recordings (Optional)

Place WAV files and a `metadata.csv` in the `custom_data/` folder on Google Drive:
```csv
file_name,transcription
rec_001.wav,A set meal
rec_002.wav,B set meal
```


In [ ]:
import os

metadata_path = f'{CUSTOM_DATA_DIR}/metadata.csv'
USE_CUSTOM_DATA = os.path.exists(metadata_path)

if USE_CUSTOM_DATA:
    import pandas as pd
    meta = pd.read_csv(metadata_path)
    print(f'Custom data: {len(meta)} samples')
    print(meta.head())

    custom_ds = load_dataset(
        'audiofolder',
        data_dir=CUSTOM_DATA_DIR,
        split='train',
    )
    print('Custom dataset columns:', custom_ds.column_names)
else:
    print(f'No custom data. Place wav + metadata.csv in: {CUSTOM_DATA_DIR}')


## 6. Preprocessing Pipeline

In streaming mode, `map()` is lazy — feature extraction runs per-batch during training.

> In Unsloth, `tokenizer` is a `WhisperProcessor` wrapper:
> - `tokenizer.feature_extractor` → mel spectrogram
> - `tokenizer.tokenizer` → BPE tokenizer


In [ ]:
from datasets import Audio, Dataset, interleave_datasets

AUDIO_COL = 'mp3'
TEXT_COL  = 'txt'
# All columns in the main dataset (used to drop originals after map)
ALL_COLS  = ['mp3', 'txt', 'json', '__key__', '__url__']

def prepare_dataset(batch):
    audio = batch[AUDIO_COL]
    features = tokenizer.feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
    )
    tokenized = tokenizer.tokenizer(batch[TEXT_COL])
    return {
        'input_features': features.input_features[0],
        'labels': tokenized.input_ids,
    }

train_ds = raw_ds['train'].cast_column(AUDIO_COL, Audio(sampling_rate=16000))

if USE_CUSTOM_DATA:
    custom_cols = custom_ds.column_names
    audio_col_src = next((c for c in custom_cols if c in ('audio', 'mp3', 'wav', 'flac')), None)
    text_col_src  = next((c for c in custom_cols if c in ('transcription', 'txt', 'text', 'sentence', 'label')), None)
    print(f'Custom: audio->{audio_col_src}, text->{text_col_src}')

    if audio_col_src and audio_col_src != AUDIO_COL:
        custom_ds = custom_ds.rename_column(audio_col_src, AUDIO_COL)
    if text_col_src and text_col_src != TEXT_COL:
        custom_ds = custom_ds.rename_column(text_col_src, TEXT_COL)

    custom_ds = custom_ds.select_columns([AUDIO_COL, TEXT_COL])
    custom_ds = custom_ds.cast_column(AUDIO_COL, Audio(sampling_rate=16000))
    custom_iter = custom_ds.to_iterable_dataset()

    # With <200 custom samples, prob=0.3 causes ~48x repetition (overfitting).
    # Target ~5 repeats: prob ≈ (5 × 200) / (2000 × 16) ≈ 0.03
    CUSTOM_PROB = 0.03
    train_ds = interleave_datasets(
        [train_ds, custom_iter],
        probabilities=[1 - CUSTOM_PROB, CUSTOM_PROB],
        stopping_strategy='all_exhausted',  # cycle custom_iter instead of stopping early
        seed=42,
    )
    print(f'Custom data interleaved (prob={CUSTOM_PROB}, stopping=all_exhausted).')

train_ds = train_ds.shuffle(buffer_size=1000, seed=42)
train_ds = train_ds.map(prepare_dataset, remove_columns=[AUDIO_COL, TEXT_COL])

# Materialize 200 eval samples to avoid streaming eval instability
print('Building eval set (skip 110000, take 200)...')
eval_stream = (
    raw_ds['train']
    .skip(110000)
    .take(200)
    .cast_column(AUDIO_COL, Audio(sampling_rate=16000))
    .map(prepare_dataset, remove_columns=ALL_COLS)
)
eval_ds = Dataset.from_list(list(eval_stream))
print(f'Preprocessing ready. Eval set: {len(eval_ds)} samples')


## 7. Data Collator & Evaluation Metric

CER (Character Error Rate) is more appropriate than WER for Chinese, which has no whitespace word boundaries.

> **Note:** `predict_with_generate=True` is disabled (following Unsloth's recommended approach).
> CER is computed via argmax over decoder logits instead of autoregressive generation.
> This means CER values may appear artificially high (e.g. 246%) — this is expected and known.
> As long as **validation loss is decreasing steadily**, training is working correctly; ignore CER.


In [ ]:
import torch
import numpy as np
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=tokenizer)

cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    # pred.predictions may be a tuple (decoder logits, ...) or a plain ndarray
    predictions = pred.predictions
    pred_logits = predictions[0] if isinstance(predictions, tuple) else predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.tokenizer.pad_token_id

    pred_ids  = np.argmax(pred_logits, axis=-1)
    pred_str  = tokenizer.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': cer}

print('Collator and metrics ready.')


## 8. Training Configuration

**Key settings:**
- `optim='adamw_8bit'` → Unsloth-optimized optimizer, reduces VRAM
- `remove_unused_columns=False` → PEFT model forward signature omits some columns; must disable
- `label_names=['labels']` → required by `Seq2SeqTrainer`
- `predict_with_generate` disabled → avoids generation-time precision issues during eval
- `max_steps` instead of `num_train_epochs` → streaming datasets have unknown length


In [ ]:
from transformers import Seq2SeqTrainingArguments
from unsloth import is_bf16_supported

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size = 16
    max_steps=2000,
    learning_rate=1e-4,
    warmup_steps=100,
    lr_scheduler_type='cosine',
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    eval_strategy='steps',
    eval_steps=200,
    per_device_eval_batch_size=4,
    # predict_with_generate=True,    # disabled: use logits argmax for CER instead
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    logging_steps=25,
    report_to=['tensorboard'],
    dataloader_num_workers=0,        # streaming + multiprocessing causes deadlocks
    push_to_hub=False,
    optim='adamw_8bit',              # Unsloth-optimized optimizer
    weight_decay=0.001,
    remove_unused_columns=False,     # required for PEFT Whisper
    label_names=['labels'],
    seed=3407,
)

use_bf16 = is_bf16_supported()
print(f'Training args configured. bf16={use_bf16}, fp16={not use_bf16}')


## 9. Train

If the session disconnects, re-run all cells — the Trainer automatically resumes from the latest checkpoint on Drive.


In [ ]:
from transformers import Seq2SeqTrainer
import glob

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer.feature_extractor,
)

# Auto-resume: pick up the latest checkpoint from Drive if available
existing_checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
resume_from = existing_checkpoints[-1] if existing_checkpoints else None

if resume_from:
    print(f'Resuming from: {resume_from}')
else:
    print('Starting fresh training.')

trainer.train(resume_from_checkpoint=resume_from)


## 10. GPU Memory Stats


In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory  = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f'GPU: {gpu_stats.name}. Max memory: {max_memory} GB.')
print(f'Peak reserved: {used_memory} GB ({round(used_memory / max_memory * 100, 1)}%)')


## 11. Save Model to Drive


In [ ]:
# Save LoRA adapters only (small, ~tens of MB)
FINAL_MODEL_DIR = f'{DRIVE_DIR}/final_model'
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f'LoRA adapters saved -> {FINAL_MODEL_DIR}')


## 12. Export to faster-whisper Format (CTranslate2)

Merge LoRA into the base model first, then convert with `ct2-transformers-converter`.


In [ ]:
import os

# Step 1: Merge LoRA into the base model (full fp16 weights)
MERGED_DIR = f'{DRIVE_DIR}/merged_model'
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved -> {MERGED_DIR}')

# Ensure config.json exists (save_pretrained_merged may omit it)
if not os.path.exists(f'{MERGED_DIR}/config.json'):
    model.config.save_pretrained(MERGED_DIR)
    print('config.json manually saved')

# Ensure tokenizer files are present
tokenizer.save_pretrained(MERGED_DIR)
print('Tokenizer files saved')

# Verify directory contents
print('merged_model contents:', sorted(os.listdir(MERGED_DIR)))

# Step 2: Convert to CTranslate2 format
!pip install -q ctranslate2

CT2_OUTPUT = f'{DRIVE_DIR}/faster_whisper_ct2'

!ct2-transformers-converter \
    --model {MERGED_DIR} \
    --output_dir {CT2_OUTPUT} \
    --copy_files tokenizer.json preprocessor_config.json \
    --quantization float16 \
    --force

print(f'faster-whisper model saved -> {CT2_OUTPUT}')
